# FeatureImpact — Notebook 6: Sequential Testing & Multiple Comparisons

**Objective:** Handle two of the most dangerous real-world A/B testing pitfalls:

### Pitfall 1: The Peeking Problem
Checking p-values repeatedly during an experiment inflates your false positive rate dramatically. A 5% significance level can become 30%+ if you peek 20 times.

### Pitfall 2: Multiple Comparisons
Testing 3 metrics at α=0.05 each gives you a ~14% chance of at least one false positive. We use the Benjamini-Hochberg (BH) correction to control the False Discovery Rate (FDR).

Both pitfalls are extremely common in industry — demonstrating awareness of them is a strong signal for any data science role.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportions_ztest

sns.set_theme(style="whitegrid")
np.random.seed(42)

df = pd.read_csv("data/ab_experiment_raw.csv", parse_dates=["entry_date"])
df = df.sort_values("entry_date").reset_index(drop=True)

print(f"Dataset: {df.shape}")
print(f"Date range: {df['entry_date'].min().date()} → {df['entry_date'].max().date()}")

## 2. The Peeking Problem — Simulating False Positive Inflation

In [ ]:
def simulate_peeking_fpr(n_simulations=5000, n_users=2000, alpha=0.05, n_peeks=20):
    """
    Simulate the false positive rate under repeated peeking.
    Under the null hypothesis (no real effect), both groups have the same conversion rate.
    """
    false_positives_peeking = 0
    false_positives_fixed   = 0

    p_null = 0.12  # Same rate for both groups (null is true)

    for _ in range(n_simulations):
        ctrl_data = np.random.binomial(1, p_null, n_users)
        trt_data  = np.random.binomial(1, p_null, n_users)

        # Fixed-horizon: test once at end
        z, p = proportions_ztest(
            [trt_data.sum(), ctrl_data.sum()],
            [n_users, n_users]
        )
        if p < alpha:
            false_positives_fixed += 1

        # Peeking: test at n_peeks evenly spaced checkpoints
        peek_points = np.linspace(50, n_users, n_peeks, dtype=int)
        peeked_positive = False
        for n in peek_points:
            z, p = proportions_ztest(
                [trt_data[:n].sum(), ctrl_data[:n].sum()],
                [n, n]
            )
            if p < alpha:
                peeked_positive = True
                break
        if peeked_positive:
            false_positives_peeking += 1

    return false_positives_fixed / n_simulations, false_positives_peeking / n_simulations

print("Simulating peeking problem (this takes ~15 seconds)...")
fpr_fixed, fpr_peeking = simulate_peeking_fpr(n_simulations=2000)

print(f"\n{'Approach':30s} | False Positive Rate")
print("-" * 55)
print(f"{'Fixed horizon (test once)':30s} | {fpr_fixed:.1%}  (target: 5%)")
print(f"{'Peeking (20 checkpoints)':30s} | {fpr_peeking:.1%}  ← INFLATED ⚠️")

## 3. Cumulative p-value Over Time — Visualising the Peeking Danger

In [ ]:
# Show how p-value fluctuates as more data accumulates (null case)
p_null = 0.12
n_total = 2000
ctrl_null = np.random.binomial(1, p_null, n_total)
trt_null  = np.random.binomial(1, p_null, n_total)  # Same rate — null is TRUE

checkpoints = list(range(50, n_total + 1, 25))
p_values_over_time = []

for n in checkpoints:
    z, p = proportions_ztest(
        [trt_null[:n].sum(), ctrl_null[:n].sum()], [n, n]
    )
    p_values_over_time.append(p)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(checkpoints, p_values_over_time, color="#E74C3C", linewidth=1.8, label="p-value over time")
ax.axhline(0.05, color="black", linestyle="--", linewidth=1.5, label="α = 0.05")

# Highlight false positive zones
for i, (n, p) in enumerate(zip(checkpoints, p_values_over_time)):
    if p < 0.05:
        ax.axvspan(n - 25, n, alpha=0.15, color="red")

ax.set_title("Cumulative p-value Under the Null (No Real Effect)\nRed zones = false positives a naive analyst might act on",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Users Observed per Group")
ax.set_ylabel("p-value")
ax.legend()
plt.tight_layout()
plt.savefig("nb6_peeking_problem.png", dpi=150, bbox_inches="tight")
plt.show()
print("If you peek whenever the p-value dips below 0.05 (red zones), you'd make the wrong call.")

## 4. Sequential Probability Ratio Test (SPRT)

SPRT lets you stop early *correctly* — it controls the false positive rate even while peeking continuously.

In [ ]:
def sprt_test(data_ctrl, data_trt, p0, p1, alpha=0.05, beta=0.20):
    """
    Sequential Probability Ratio Test.
    p0: null hypothesis conversion rate
    p1: alternative hypothesis conversion rate (MDE)
    Returns: log-likelihood ratios and stopping point.
    """
    A = np.log((1 - beta) / alpha)      # Upper threshold: reject H0
    B = np.log(beta / (1 - alpha))      # Lower threshold: accept H0

    log_llr = 0
    llr_history = []
    stopping_point = None
    decision = None

    n = min(len(data_ctrl), len(data_trt))
    for i in range(n):
        x_c = data_ctrl[i]
        x_t = data_trt[i]

        # Log likelihood ratio update
        llr_ctrl = x_c * np.log(p0/(p0)) + (1-x_c) * np.log((1-p0)/(1-p0))  # = 0 baseline
        llr_trt  = (x_t * np.log(p1/p0) + (1-x_t) * np.log((1-p1)/(1-p0)))
        log_llr += llr_trt
        llr_history.append(log_llr)

        if log_llr >= A and stopping_point is None:
            stopping_point = i + 1
            decision = "REJECT H₀ (ship treatment)"
            break
        elif log_llr <= B and stopping_point is None:
            stopping_point = i + 1
            decision = "ACCEPT H₀ (keep control)"
            break

    return llr_history, stopping_point, decision, A, B


# Use actual experiment data
control   = df[df["group"] == "control"].reset_index(drop=True)
treatment = df[df["group"] == "treatment"].reset_index(drop=True)

ctrl_conv = control["converted"].values
trt_conv  = treatment["converted"].values

llr_hist, stop_pt, decision_sprt, A_thresh, B_thresh = sprt_test(
    ctrl_conv, trt_conv, p0=0.12, p1=0.15, alpha=0.05, beta=0.20
)

print(f"SPRT Result:")
print(f"  Decision       : {decision_sprt}")
print(f"  Stopping point : {stop_pt:,} users observed" if stop_pt else "  No stopping point reached — collect more data")
print(f"  Upper threshold (A): {A_thresh:.3f}")
print(f"  Lower threshold (B): {B_thresh:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(1, len(llr_hist)+1), llr_hist, color="#2980B9", linewidth=1.5, label="Log-likelihood ratio")
ax.axhline(A_thresh, color="#2ECC71", linestyle="--", linewidth=2, label=f"Upper bound A={A_thresh:.2f} (reject H₀)")
ax.axhline(B_thresh, color="#E74C3C", linestyle="--", linewidth=2, label=f"Lower bound B={B_thresh:.2f} (accept H₀)")

if stop_pt:
    ax.axvline(stop_pt, color="black", linestyle=":", linewidth=2, label=f"Stop at n={stop_pt:,}")

ax.set_title("Sequential Probability Ratio Test (SPRT)\nStatistically valid early stopping", fontsize=12, fontweight="bold")
ax.set_xlabel("Users Observed")
ax.set_ylabel("Log-Likelihood Ratio")
ax.legend()
plt.tight_layout()
plt.savefig("nb6_sprt.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Multiple Comparisons — Benjamini-Hochberg Correction

In [ ]:
# Raw p-values from Notebook 3
from statsmodels.stats.proportion import proportions_ztest

control   = df[df["group"] == "control"]
treatment = df[df["group"] == "treatment"]
df_clean  = df.dropna(subset=["session_duration_sec"])

n_ctrl, n_trt = len(control), len(treatment)

_, p_conv = proportions_ztest(
    [treatment["converted"].sum(), control["converted"].sum()], [n_trt, n_ctrl]
)
_, p_sess = stats.ttest_ind(
    df_clean[df_clean["group"]=="treatment"]["session_duration_sec"],
    df_clean[df_clean["group"]=="control"]["session_duration_sec"],
    equal_var=False
)
_, p_ctr, _, _ = stats.chi2_contingency(np.array([
    [treatment["clicked_prompt"].sum(), n_trt - treatment["clicked_prompt"].sum()],
    [control["clicked_prompt"].sum(),  n_ctrl - control["clicked_prompt"].sum()]
]))

metrics   = ["Conversion Rate", "Session Duration", "Click-Through Rate"]
raw_pvals = [p_conv, p_sess, p_ctr]

# Apply Benjamini-Hochberg correction
reject, pvals_corrected, _, _ = multipletests(raw_pvals, alpha=0.05, method="fdr_bh")

results = pd.DataFrame({
    "Metric":        metrics,
    "Raw p-value":   [f"{p:.6f}" for p in raw_pvals],
    "BH-corrected":  [f"{p:.6f}" for p in pvals_corrected],
    "Significant (raw)": [p < 0.05 for p in raw_pvals],
    "Significant (BH)":  list(reject)
})

print("Multiple Comparisons Correction — Benjamini-Hochberg (FDR)")
print("=" * 70)
print(results.to_string(index=False))
print("\nNote: BH correction controls the False Discovery Rate across all 3 tests.")

## 6. Visualise Corrected vs Uncorrected p-values

In [ ]:
x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, raw_pvals,       width, label="Raw p-values",  color="#E74C3C", alpha=0.8)
bars2 = ax.bar(x + width/2, pvals_corrected, width, label="BH corrected",  color="#2980B9", alpha=0.8)

ax.axhline(0.05, color="black", linestyle="--", linewidth=1.5, label="α = 0.05")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel("p-value")
ax.set_title("Raw vs Benjamini-Hochberg Corrected p-values", fontsize=12, fontweight="bold")
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f"{bar.get_height():.4f}", ha="center", fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f"{bar.get_height():.4f}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("nb6_multiple_comparisons.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

| Pitfall | Problem | Solution Used |
|---|---|---|
| Peeking | Inflated false positive rate | SPRT for valid early stopping |
| Multiple metrics | Family-wise error inflation | Benjamini-Hochberg FDR correction |

**Next:** Notebook 7 — Auto Report Generation (turning all analysis into a shareable PDF/HTML report).